# 作业 4

- 姓名：丁平尖
- 日期：2026 年 6 月 17 日

## 1 注意事项
1. 编程题需要打印相应的输出。
2. 将 ipynb 文件提交至 github 上，命名为：HW04-学号-姓名.ipynb。

# 2 序列模型

## 2.1 理论计算题
给定一个字符序列 "ababc"，假设采用一阶马尔可夫模型 $p(x_t \mid x_{t-1})$，使用拉普拉斯平滑（加 1 平滑）估计以下条件概率：
1. $p(a \mid b)$
2. $p(c \mid b)$

词汇表为 $\{a, b, c\}$，计算时考虑所有可能转移，包括未出现的情况。

解：序列 "ababc" 的相邻转移为：
- a -> b
- b -> a
- a -> b
- b -> c

从字符 b 出发的转移统计为：
- count(b -> a) = 1
- count(b -> b) = 0
- count(b -> c) = 1

使用拉普拉斯平滑：
$$
p(x_t = w \mid x_{t-1} = b) = \frac{count(b \to w) + 1}{count(b \to a) + count(b \to b) + count(b \to c) + V}
$$

其中 $V = 3$，因此：
$$
p(a \mid b) = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4
$$
$$
p(c \mid b) = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4
$$

补充：
$$
p(b \mid b) = \frac{0 + 1}{2 + 3} = \frac{1}{5} = 0.2
$$

## 2.2 编程题
编写一个函数 `preprocess_text(text, n)`，完成以下步骤：
1. 将文本转换为小写，去除标点符号，仅保留字母和空格。
2. 按空格分词。
3. 构建词汇表，按出现频率排序并分配整数 ID，从 0 开始。
4. 使用滑动窗口生成长度为 `n` 的特征序列和对应的下一个词标签，用于自回归语言模型。

返回词汇表字典和 `(特征列表, 标签列表)`。示例：输入 "The time machine" 和 `n=2`，应生成特征 `[['the', 'time'], ['time', 'machine']]`，标签 `['machine', None]`；若无后续词则忽略。

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()

    vocab_counts = Counter(tokens)
    vocab = {word: idx for idx, (word, _) in enumerate(sorted(vocab_counts.items(), key=lambda item: (-item[1], item[0])))}

    features = []
    labels = []
    for i in range(len(tokens) - n):
        features.append(tokens[i:i + n])
        labels.append(tokens[i + n])

    return vocab, features, labels

sample_text = "The time machine"
vocab, features, labels = preprocess_text(sample_text, 2)
print('vocab:', vocab)
print('features:', features)
print('labels:', labels)

vocab: {'machine': 0, 'the': 1, 'time': 2}
features: [['the', 'time']]
labels: ['machine']


# 3 循环神经网络

## 3.1 理论计算题
考虑一个线性 RNN（无偏置），定义为
$$
h_t = W_{hh} h_{t-1} + W_{hx} x_t,
$$
输出为
$$
o_t = W_o h_t,
$$
假设损失函数为平方损失
$$
L = \frac{1}{2} \sum_{t=1}^{T} (o_t - y_t)^2.
$$
推导损失对权重 $W_{hh}$ 的梯度表达式（通过时间反向传播，展开到所有时间步），并说明梯度消失或爆炸的条件。

解：设 $\delta_t = \partial L / \partial h_t$，$\epsilon_t = \partial L / \partial o_t = o_t - y_t$。
由 $o_t = W_o h_t$ 得：
$$
\delta_t = W_o^\top \epsilon_t + W_{hh}^\top \delta_{t+1}, \quad \delta_{T+1}=0
$$

因此，
$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \delta_t h_{t-1}^\top
$$

进一步展开可写成：
$$
\delta_t = \sum_{k=t}^{T} (W_{hh}^\top)^{k-t} W_o^\top (o_k - y_k)
$$

梯度在时间步上传递时会反复乘以 $W_{hh}^\top$。
当 $\rho(W_{hh}) < 1$ 时容易梯度消失；当 $\rho(W_{hh}) > 1$ 时容易梯度爆炸；当谱半径接近 1 时，相对更稳定。

## 3.2 编程题
实现一个简单的 RNN 单元的前向传播和单步反向传播（仅计算梯度，不更新参数）。
给定输入 $x_t$（形状 $(batch\_size, input\_size)$）、上一隐藏状态 $h_{prev}$（形状 $(batch\_size, hidden\_size)$），以及权重 $W_{hx}$、$W_{hh}$、$b_h$，计算当前隐藏状态 $h_t$。

同时实现反向传播：已知上游梯度 $dh_{next}$（即损失对 $h_t$ 的梯度），计算 $dx_t$、$dh_{prev}$、$dW_{hx}$、$dW_{hh}$、$db_h$，使用 tanh 激活函数。

In [1]:
try:
    import torch
except ModuleNotFoundError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch"])
    import torch

def rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h):
    preact = x_t @ W_hx.T + h_prev @ W_hh.T + b_h
    h_t = torch.tanh(preact)
    cache = (x_t, h_prev, W_hx, W_hh, h_t)
    return h_t, cache

def rnn_step_backward(dh_next, cache):
    x_t, h_prev, W_hx, W_hh, h_t = cache
    dtanh = dh_next * (1 - h_t.pow(2))
    dx_t = dtanh @ W_hx
    dh_prev = dtanh @ W_hh
    dW_hx = dtanh.T @ x_t
    dW_hh = dtanh.T @ h_prev
    db_h = dtanh.sum(dim=0)
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

batch_size, input_size, hidden_size = 2, 3, 4
x_t = torch.randn(batch_size, input_size)
h_prev = torch.randn(batch_size, hidden_size)
W_hx = torch.randn(hidden_size, input_size)
W_hh = torch.randn(hidden_size, hidden_size)
b_h = torch.randn(hidden_size)

h_t, cache = rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h)
dh_next = torch.randn_like(h_t)
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_step_backward(dh_next, cache)

print('h_t shape:', tuple(h_t.shape))
print('dx_t shape:', tuple(dx_t.shape))
print('dh_prev shape:', tuple(dh_prev.shape))
print('dW_hx shape:', tuple(dW_hx.shape))
print('dW_hh shape:', tuple(dW_hh.shape))
print('db_h shape:', tuple(db_h.shape))

h_t shape: (2, 4)
dx_t shape: (2, 3)
dh_prev shape: (2, 4)
dW_hx shape: (4, 3)
dW_hh shape: (4, 4)
db_h shape: (4,)


# 4 高级循环神经网络

## 4.1 理论计算题
假设一个深度双向 RNN，有 $L$ 层，每层隐藏单元数为 $H$，输入维度为 $D$，输出维度为 $O$（仅考虑最后输出层）。
计算该模型的参数总数（包括所有全连接层的权重和偏置），忽略嵌入层和输出层之前的投影，明确给出表达式。

解：若每层都是双向 RNN，则第 1 层每个方向的参数为：
$$
D H + H^2 + H
$$

所以第 1 层双向总参数为：
$$
2(DH + H^2 + H)
$$

从第 2 层开始，每层输入维度变为 $2H$，因此每层双向参数为：
$$
2(2H\cdot H + H^2 + H) = 2(3H^2 + H)
$$

若共有 $L$ 层，则双向 RNN 主体参数总数为：
$$
2(DH + H^2 + H) + (L-1)\,2(3H^2 + H)
$$

若最后还有一个输出全连接层从 $2H$ 映射到 $O$，则再加：
$$
2HO + O
$$

最终总参数量：
$$
2(DH + H^2 + H) + (L-1)\,2(3H^2 + H) + 2HO + O
$$

## 4.2 编程题
实现一个双向 RNN 编码器，接收序列 $X$（形状 $(seq\_len, batch, input\_dim)$），使用 `torch.nn.RNN` 或手动实现。
要求返回每个时间步的拼接后的前向和后向隐藏状态（形状 $(seq\_len, batch, 2*hidden\_dim)$），以及最终时间步的拼接隐藏状态（作为序列表示）。


```python
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True
        )

    def forward(self, X):
        outputs, h_n = self.rnn(X)
        final_state = torch.cat((h_n[-2], h_n[-1]), dim=-1)
        return outputs, final_state
```


In [3]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True
        )

    def forward(self, X):
        outputs, h_n = self.rnn(X)
        final_state = torch.cat((h_n[-2], h_n[-1]), dim=-1)
        return outputs, final_state

encoder = BiRNNEncoder(input_dim=5, hidden_dim=4)
X = torch.randn(6, 2, 5)
outputs, seq_repr = encoder(X)
print('outputs shape:', tuple(outputs.shape))
print('sequence representation shape:', tuple(seq_repr.shape))

outputs shape: (6, 2, 8)
sequence representation shape: (2, 8)


# 5 嵌入向量

## 5.1 理论计算题
在 Skip-gram 模型中，给定中心词 $w_c$ 和上下文词 $w_o$，使用负采样（采样 $K$ 个负样本）。
推导其损失函数（对数似然）的表达式，并说明如何从噪声分布中采样负样本。
假设词向量为 $v_c, u_o$，负样本词向量为 $u_{n_k}$，写出完整的目标函数。



解：Skip-gram 结合负采样的目标函数可写为：
$$
\log \sigma(v_c^\top u_o) + \sum_{k=1}^{K} \mathbb{E}_{n_k \sim P_n(w)} \left[ \log \sigma(-v_c^\top u_{n_k}) \right]
$$

其中：
- $v_c$ 表示中心词向量；
- $u_o$ 表示正样本上下文词向量；
- $u_{n_k}$ 表示第 $k$ 个负样本词向量；
- $P_n(w)$ 是噪声分布，常用词频的 $3/4$ 次幂分布。

训练时，负样本从噪声分布中独立采样，通常排除真实上下文词本身。

## 5.2 编程题
实现 CBOW 模型的前向传播和损失计算（不使用负采样，仅用完整 softmax）。
给定一批上下文词的索引列表（每个样本有 `context_size` 个上下文词），词汇表大小 $V$，嵌入维度 $d$。
输入权重矩阵 $W$（形状 $(V, d)$）和输出权重矩阵 $W_{out}$（形状 $(d, V)$）。
计算平均上下文向量作为隐藏层，然后计算输出概率分布，并计算交叉熵损失（目标为中心词索引）。返回损失值。

```python
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, center_indices, W, W_out):
    context_embeds = W[context_indices]
    hidden = context_embeds.mean(dim=1)
    logits = hidden @ W_out
    probs = F.softmax(logits, dim=-1)
    loss = F.cross_entropy(logits, center_indices)
    return loss, probs
```


In [5]:
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, center_indices, W, W_out):
    context_embeds = W[context_indices]
    hidden = context_embeds.mean(dim=1)
    logits = hidden @ W_out
    probs = F.softmax(logits, dim=-1)
    loss = F.cross_entropy(logits, center_indices)
    return loss, probs

V, d = 8, 4
W = torch.randn(V, d, requires_grad=True)
W_out = torch.randn(d, V, requires_grad=True)
context_indices = torch.tensor([[0, 1, 2], [3, 4, 5]])
center_indices = torch.tensor([6, 7])

loss, probs = cbow_forward(context_indices, center_indices, W, W_out)
print('loss:', float(loss))
print('probs shape:', tuple(probs.shape))

loss: 1.9579026699066162
probs shape: (2, 8)


# 6 注意力机制

## 6.1 理论计算题
给定查询矩阵 $Q \in R^{2\times4}$，键矩阵 $K \in R^{3\times4}$，值矩阵 $V \in R^{3\times5}$，计算缩放点积注意力（无掩码）的输出矩阵。

要求写出中间步骤：
1. 先计算得分矩阵；
2. 再进行 softmax；
3. 最后做加权求和。

使用公式：$score = QK^T / \sqrt{d_k}$，其中 $d_k = 4$。
可以只列出数值计算过程，符号或具体数值均可。

解：缩放点积注意力先计算得分矩阵：
$$
S = \frac{QK^\top}{\sqrt{d_k}} = \frac{QK^\top}{2}
$$

其中 $S \in \mathbb{R}^{2\times3}$。
然后对每一行做 softmax：
$$
A = \text{softmax}(S)
$$

得到注意力权重矩阵 $A \in \mathbb{R}^{2\times3}$。
最后加权求和：
$$
O = AV
$$

因此输出矩阵 $O \in \mathbb{R}^{2\times5}$。

## 6.2 编程题
实现多头注意力（Multi-Head Attention）的前向传播。

设定：
- `num_heads = 2`
- `d_model = 4`

给定输入 $X$，其形状为 $(seq\_len, batch, d\_model)$。
先分别通过线性层投影得到 $Q$、$K$、$V$，其中每个头的维度为 $d_k = d_v = d\_model / num\_heads$。

对每个头分别计算缩放点积注意力，再将所有头的输出拼接，最后经过一层线性映射。
输出张量的形状应与输入保持一致。

In [6]:
import math
import torch
import torch.nn as nn
import math
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, X):
        seq_len, batch_size, _ = X.shape
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)

        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        V = V.view(seq_len, batch_size, self.num_heads, self.d_k).permute(1, 2, 0, 3)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)

        context = context.permute(2, 0, 1, 3).contiguous().view(seq_len, batch_size, self.d_model)
        output = self.W_o(context)
        return output
    
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, X):
        seq_len, batch_size, _ = X.shape
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)

        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        V = V.view(seq_len, batch_size, self.num_heads, self.d_k).permute(1, 2, 0, 3)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)

        context = context.permute(2, 0, 1, 3).contiguous().view(seq_len, batch_size, self.d_model)
        output = self.W_o(context)
        return output

mha = MultiHeadAttention(d_model=4, num_heads=2)
X = torch.randn(5, 3, 4)
Y = mha(X)
print('output shape:', tuple(Y.shape))

output shape: (5, 3, 4)
